# 🦀 Day 4: Procedural Macros — Intro
---

Today we introduce **procedural macros** — Rust functions that transform `TokenStream`s.

- **What proc macros are**: Functions that take and return `TokenStream`
- **Three types**: Derive, attribute, function-like
- **How they differ from macro_rules!**: Arbitrary Rust code, full AST access
- **Proc macro crate setup**: `proc-macro = true` in Cargo.toml
- **Key crates**: `proc-macro2`, `syn`, `quote`
- **Project structure**: Separate crate required

## Three Types of Procedural Macros

| Type | Example | Purpose |
|------|---------|----------|
| **Derive** | `#[derive(HelloMacro)]` | Add trait impls, methods |
| **Attribute** | `#[log_calls]` | Modify/replace annotated item |
| **Function-like** | `hello!()` | Custom macro invocation |

All receive `TokenStream` input and produce `TokenStream` output. They run at compile time.

In [ ]:
// Proc macros require a SEPARATE crate — they cannot live in the same crate as users.
// Cargo.toml for the proc-macro crate:
// [lib]
// proc-macro = true
//
// [dependencies]
// proc-macro2 = "1"
// syn = { version = "2", features = ["full"] }
// quote = "1"

// lib.rs — function-like proc macro skeleton:
// use proc_macro::TokenStream;
//
// #[proc_macro]
// pub fn hello(input: TokenStream) -> TokenStream {
//     // input is the tokens inside hello!(...)
//     // Return new TokenStream (generated code)
//     input
// }

// Derive macro skeleton:
// #[proc_macro_derive(HelloMacro)]
// pub fn hello_macro_derive(input: TokenStream) -> TokenStream {
//     let ast = syn::parse_macro_input!(input as syn::DeriveInput);
//     let name = &ast.ident;
//     let gen = quote::quote! {
//         impl HelloMacro for #name {
//             fn hello() { println!("Hello, Macro! My name is {}!", stringify!(#name)); }
//         }
//     };
//     gen.into()
// }

println!("Proc macros need a separate crate — see project structure above.");

## Complete Project Structure

```
my_macros/
├── Cargo.toml          # [lib] proc-macro = true
├── src/
│   └── lib.rs         # proc macro implementations
└── examples/
    └── hello.rs       # uses the macro

my_app/
├── Cargo.toml         # depends on my_macros
└── src/main.rs        # use my_macros::HelloMacro;
```

In [ ]:
// Example: #[derive(HelloMacro)] adds a hello() method
// In the proc-macro crate:
//
// #[proc_macro_derive(HelloMacro)]
// pub fn hello_macro_derive(input: TokenStream) -> TokenStream {
//     let ast = syn::parse_macro_input!(input as syn::DeriveInput);
//     let name = ast.ident;
//     quote::quote! {
//         impl HelloMacro for #name {
//             fn hello() {
//                 println!("Hello, Macro! My name is {}!", stringify!(#name));
//             }
//         }
//     }.into()
// }
//
// In user code:
// #[derive(HelloMacro)]
// struct Pancakes;
// Pancakes::hello();  // prints "Hello, Macro! My name is Pancakes!"

println!("Design complete — implement in a proc-macro crate.");

## 📝 Exercise: Design a Validate derive macro

On paper (or in markdown), design a `#[derive(Validate)]` macro that:
- Generates a `validate() -> Result<(), String>` method
- For each field, checks non-empty for `String`, positive for `u32`, etc.
- Returns `Ok(())` if all valid, or `Err("field X: reason")` otherwise

What would the `syn::DeriveInput` look like? What would `quote!` generate?

In [ ]:
// YOUR CODE HERE — design in comments:
// struct User {
//     name: String,
//     age: u32,
// }
// #[derive(Validate)]
// impl Validate for User { fn validate(&self) -> Result<(), String> { ... } }
//
// What fields does DeriveInput have? How do we iterate struct fields?

## 🎯 Key Takeaways

1. **Proc macros** are Rust functions: `TokenStream -> TokenStream`
2. **Three types**: derive, attribute, function-like — each has a different signature
3. They **must** live in a separate crate with `proc-macro = true`
4. **syn** parses Rust code into AST; **quote** generates Rust code from templates
5. **TokenStream** and **TokenTree** are the raw representation of tokens
6. Derive macros use `syn::parse_macro_input!(input as DeriveInput)` to get the struct/enum

---
**Tomorrow:** Derive macros with syn and quote